# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. `20 10` — `s["b"]` берёт по метке (b → 20), `s.iloc[0]` по позиции (первый элемент → 10).
2. `Алматы NaN`, `Астана NaN`, `Шымкент 3.0` — Pandas складывает по меткам индекса, а не по позициям. Совпала только метка «Шымкент» (2 + 1 = 3). У «Алматы» и «Астана» нет пары в другой Series, поэтому NaN; из-за NaN тип становится float.
3. `(2, 2)` — условие `x > 1` оставляет строки с x = 2 и x = 3, столбцов по-прежнему 2.
4. `Series DataFrame` — одинарные скобки `df["x"]` дают Series (один столбец), двойные `df[["x"]]` дают DataFrame (таблицу из одного столбца).
5. `[1, 2] 5` — фильтр сохраняет исходные метки строк (1 и 2), а не пересчитывает их с нуля. `f.iloc[0, 1]` это первая строка отфильтрованной таблицы, столбец `y`: значение 5.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

In [ ]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [ ]:
# 1
print("Строк и столбцов:", df.shape)
print("\nТипы столбцов:")
print(df.dtypes)
print("\nПропуски по столбцам:")
print(df.isna().sum())
print("\nПропусков в delivery_days:", df["delivery_days"].isna().sum())
print("Статусы заказов с пропуском срока доставки:")
print(df.loc[df["delivery_days"].isna(), "status"].value_counts())

print("\nРазных значений:")
for col in ["city", "category", "status", "customer_id"]:
    print(f"  {col}: {df[col].nunique()}")
print("\nВсе написания города:")
print(df["city"].value_counts())
print("\nСтатусы:")
print(df["status"].value_counts())

n_dup = df["order_id"].duplicated().sum()
n_test = (~not_test).sum()
dup_rev = df.loc[df["order_id"].duplicated() & paid & not_test, "amount"].sum()
test_rev = df.loc[paid & ~not_test, "amount"].sum()
print("\nПовторяющихся order_id (лишних строк):", n_dup, "| строк клиента TEST:", n_test)
print(f"Выручка, которая приходится на лишние дубли: {dup_rev:,.0f} ₸ | на оплаченные заказы TEST: {test_rev:,.0f} ₸")

print(f"""
Менеджеру: в данных три проблемы.
1. Город записан по-разному: {df['city'].nunique()} написаний вместо шести городов. Отчёт по городам разобьёт один город на несколько строк и занизит каждый.
2. Дубли заказов: {n_dup} лишних строк с повторяющимся order_id. Выручка завышена примерно на {dup_rev:,.0f} ₸.
3. Тестовый клиент TEST: {n_test} строк, оплаченных на {test_rev:,.0f} ₸. Это не покупатели, их нужно исключать из выручки.
Дополнительно: в delivery_days {df['delivery_days'].isna().sum()} пропусков, средний срок доставки считать только по заполненным.""")

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [ ]:
# 2
x = df.loc[paid & not_test, "amount"]
mean, med = x.mean(), x.median()
q1, q3 = x.quantile(0.25), x.quantile(0.75)
print(f"Заказов: {len(x)}")
print(f"Среднее: {mean:,.0f} ₸ | медиана: {med:,.0f} ₸ | Q1: {q1:,.0f} ₸ | Q3: {q3:,.0f} ₸")
print(f"Среднее выше медианы в {mean/med:.2f} раза")
print(f"""Менеджеру: типичный чек — {med:,.0f} ₸, половина заказов от {q1:,.0f} до {q3:,.0f} ₸.
В отчёте честнее показать медиану: среднее ({mean:,.0f} ₸) {'тянут вверх редкие очень дорогие заказы' if mean > med else 'почти не отличается от медианы'}, и оно описывает не типичного клиента, а общий итог, делённый на число заказов.""")

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [ ]:
# 3
cancel = df["status"] == "отменён"

# списки написаний собираем из самих данных (обычные строковые методы Python, не .str из Pandas)
AL = [c for c in df["city"].unique() if c.strip().lower() in ("алматы", "almaty")]
SH_data = [c for c in df["city"].unique() if c.strip().lower() in ("шымкент", "shymkent")]
print("Написания Алматы:", AL)
print("Написания Шымкента в данных:", SH_data, "| в списке SH:", SH)

share_all = cancel.mean()
share_sh = cancel[df["city"].isin(SH)].mean()
share_al = cancel[df["city"].isin(AL)].mean()
print(f"\nДоля отмен: все заказы {share_all:.1%} | Шымкент {share_sh:.1%} | Алматы {share_al:.1%}")
print(f"Шымкент выше среднего в {share_sh/share_all:.1f} раза, выше Алматы в {share_sh/share_al:.1f} раза")
top_city, top_share, other_city, other_share = ("Шымкент", share_sh, "Алматы", share_al) if share_sh >= share_al else ("Алматы", share_al, "Шымкент", share_sh)
print(f"Менеджеру: выше отмены в городе {top_city}: {top_share:.1%} против {other_share:.1%} в городе {other_city} и {share_all:.1%} в среднем по всем заказам "
      f"(в {top_share/other_share:.1f} раза выше, чем в {other_city}). Причины отмен (доставка, оплата, наличие товара) нужно смотреть сначала там.")

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [ ]:
# 4
base = paid & not_test            # дубли пока не убираем (по условию)
AL = [c for c in df["city"].unique() if c.strip().lower() in ("алматы", "almaty")]

def sep_oct(city_list):
    m = base & df["city"].isin(city_list)
    s = df.loc[m & sep, "amount"].sum()
    o = df.loc[m & octo, "amount"].sum()
    return s, o, (o - s) / s * 100

s_sh, o_sh, c_sh = sep_oct(SH)
s_al, o_al, c_al = sep_oct(AL)
print(f"Шымкент: сентябрь {s_sh:,.0f} ₸ -> октябрь {o_sh:,.0f} ₸ ({c_sh:+.1f}%)")
print(f"Алматы:  сентябрь {s_al:,.0f} ₸ -> октябрь {o_al:,.0f} ₸ ({c_al:+.1f}%)")

worse, better = ("Шымкент", "Алматы") if c_sh < c_al else ("Алматы", "Шымкент")
c_w, c_b = (c_sh, c_al) if worse == "Шымкент" else (c_al, c_sh)
ratio = f"в {abs(c_w)/abs(c_b):.1f} раза" if c_w < 0 and c_b < 0 else "по-разному (у одного падение, у другого рост)"
print(f"Динамика хуже в городе {worse}: {c_w:+.1f}% против {c_b:+.1f}% у города {better} ({ratio}).")
print(f"Менеджеру: выручка {worse} изменилась на {c_w:+.1f}%, {better} на {c_b:+.1f}%. "
      "Цифры предварительные: дубли заказов не убраны, а лишние написания городов могли не попасть в список. "
      "Сначала нужно очистить данные, потом делать вывод о причине.")

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [ ]:
# 5
# Ошибка 1 (код падает): оператор & приоритетнее ==, поэтому Python сначала считает
#   "Шымкент" & df["status"]  ->  TypeError. Каждое сравнение нужно взять в скобки.
# Ошибка 2 (тихо неверное число): df["city"] == "Шымкент" ловит только одно написание;
#   «шымкент» и «Shymkent» теряются. Нужно isin(SH).
try:
    bad = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
except TypeError as e:
    print("Оригинальный код падает:", e)

only_exact = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]    # скобки исправили, но число занижено
cancel_sh = df[df["city"].isin(SH) & (df["status"] == "отменён")]          # исправлено полностью
print("Со скобками, но только «Шымкент»:", len(only_exact))
print("Исправленный код (все написания):", len(cancel_sh))
print(f"Менеджеру: в Шымкенте отменено {len(cancel_sh)} заказов; "
      f"исходная версия недосчитала бы {len(cancel_sh) - len(only_exact)} из-за других написаний города.")

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [ ]:
# 6
vc = df["customer_id"].value_counts()
print("Топ-5 клиентов по числу заказов:")
print(vc.head(5))

top = vc.index[vc.index != "TEST"][0]            # самый частый реальный клиент
print("\nСамый частый реальный клиент:", top, "|", vc[top], "заказов")
# order_id добавлен к нужным столбцам, чтобы проверить дубли
orders = df.loc[df["customer_id"] == top, ["order_id", "order_date", "city", "amount", "status"]]
print(orders.to_string())

dup_in = orders["order_id"].duplicated(keep=False).sum()
n_dup = df["order_id"].duplicated().sum()
dup_rev = df.loc[df["order_id"].duplicated() & paid & not_test, "amount"].sum()
all_rev = df.loc[paid & not_test, "amount"].sum()
print(f"\nСтрок с повторяющимся order_id у этого клиента: {dup_in}")
print(f"Во всей таблице лишних дублей: {n_dup}, они дают {dup_rev:,.0f} ₸ из {all_rev:,.0f} ₸ ({dup_rev/all_rev:.1%}) выручки")
print(f"Менеджеру: {'у клиента ' + top + ' есть заказы-дубли, ' if dup_in else 'у клиента ' + top + ' дублей нет, но '}"
      f"по всей таблице выручка, которую мы считали на уроке, завышена на {dup_rev:,.0f} ₸ ({dup_rev/all_rev:.1%}) из-за повторяющихся order_id. "
      "Перед отчётом дубли нужно убрать.")
if vc.index[0] == "TEST":
    print("Примечание: больше всего заказов у тестового клиента TEST, поэтому в разборе взят самый частый реальный клиент.")

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7
x = df.loc[paid & not_test, "amount"]
q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr = q3 - q1
bound = q3 + 1.5 * iqr
out_mask = paid & not_test & (df["amount"] > bound)
n_out = out_mask.sum()
print(f"Q1 = {q1:,.0f} ₸, Q3 = {q3:,.0f} ₸, IQR = {iqr:,.0f} ₸, граница = {bound:,.0f} ₸")
print(f"Выше границы: {n_out} заказов, {n_out/len(x):.1%} от оплаченных без TEST")
cats = df.loc[out_mask, "category"].value_counts()
print("Категории среди выбросов:")
print(cats)

# для сравнения: z-оценка на этой же выборке (дубли не убраны)
z = (x - x.mean()) / x.std(ddof=0)
n_z = (z.abs() > 3).sum()
print(f"\nНа этой же выборке |z| > 3: {n_z} заказов (в ДЗ 4 было 87 на данных без дублей и без TEST)")
print(f"Менеджеру: {n_out} заказов ({n_out/len(x):.1%}) дороже {bound:,.0f} ₸ необычно велики, чаще всего это категория «{cats.index[0]}».")
print(f"Почему правила дают разное число (IQR: {n_out}, z-оценка: {n_z}):")
print("""
- IQR опирается на квартили, а z-оценка на среднее и std. Крупные заказы сами раздувают среднее и std, и z-порог 3 отодвигается дальше (эффект маскировки).
- Границы разные: Q3 + 1.5·IQR обычно лежит ближе к основной массе заказов, чем среднее + 3·std, особенно при правостороннем распределении сумм.
- Правило IQR здесь смотрит только вверх, z-оценка |z| > 3 считает выбросы и сверху, и снизу.
- Выборки могут отличаться: в ДЗ 4 данные были очищены от дублей, здесь дубли ещё остаются.""")

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8
cancel_by_city = df.loc[df["status"] == "отменён", "city"].value_counts()
all_by_city = df["city"].value_counts()
print("Порядок меток совпадает в двух Series:", list(cancel_by_city.index) == list(all_by_city.index))

# Деление идёт по меткам индекса, поэтому порядок не важен; города без отмен дают NaN, заменяем на 0
rate = (cancel_by_city / all_by_city).fillna(0).sort_values(ascending=False)

# проверка вручную для одного написания
c0 = rate.index[0]
manual = cancel_by_city.get(c0, 0) / all_by_city[c0]
print(f"Проверка для «{c0}»: вручную {manual:.4f}, Pandas {rate[c0]:.4f}, совпало: {np.isclose(manual, rate[c0])}")

res = pd.DataFrame({"заказов": all_by_city, "отменено": cancel_by_city.reindex(all_by_city.index, fill_value=0), "доля отмен": rate})
print(res.sort_values("доля отмен", ascending=False))

sh_all = df["city"].isin(SH)
print(f"\nВсе написания Шымкента вместе: {sh_all.sum()} заказов, доля отмен {(df['status'] == 'отменён')[sh_all].mean():.1%}")
print("""Менеджеру: судить о городе по одному написанию нельзя. Данные по городу разбиты на несколько строк, у редких написаний заказов мало,
и их доля отмен случайно скачет (несколько заказов дают большой процент). Перед анализом нужно привести города к одному написанию
(регистр, пробелы, латиница и кириллица), как на уроке 6, и считать долю по объединённому городу.""")